
## Background

In this tutorial, you will run an experiment on a quantum computer to demonstrate the violation of the CHSH inequality with the Estimator primitive.

The CHSH inequality, named after the authors Clauser, Horne, Shimony, and Holt, is used to experimentally prove Bell's theorem (1969). This theorem asserts that local hidden variable theories cannot account for some consequences of entanglement in quantum mechanics. The violation of the CHSH inequality is used to show that quantum mechanics is incompatible with local hidden-variable theories. This is an important experiment for understanding the foundation of quantum mechanics.

The 2022 Nobel Prize for Physics was awarded to Alain Aspect, John Clauser and Anton Zeilinger in part for their pioneering work in quantum information science, and in particular, for their experiments with entangled photons demonstrating violation of Bell’s inequalities.


## Step 1. Map classical inputs to a quantum problem


For this experiment, we will create an entangled pair on which we measure each qubit on two different bases. We will label the bases for the first qubit $A$ and $a$ and the bases for the second qubit $B$ and $b$.  This allows us to compute the CHSH quantity $S_1$:

$$
S_1 = A(B-b) + a(B+b).
$$

Each observable is either $+1$ or $-1$. Clearly, one of the terms $B\pm b$ must be $0$, and the other must be $\pm 2$.  Therefore, $S_1 = \pm 2$. The average value of $S_1$ must satisfy the inequality:

$$
|\langle S_1 \rangle|\leq 2.
$$

Expanding $S_1$ in terms of $A$, $a$, $B$, and $b$ results in:

$$
|\langle S_1 \rangle| = |\langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle| \leq 2
$$

You can define another CHSH quantity $S_2$:

$$
S_2 = A(B+b) - a(B-b),
$$

This leads to another inequality:

$$
|\langle S_2 \rangle| = |\langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle| \leq 2
$$

If quantum mechanics can be described by local hidden variable theories, the previous inequalities must hold true. However, as is demonstrated in this notebook, these inequalities can be violated in a quantum computer.  Therefore, quantum mechanics is not compatible with local hidden variable theories.


You will create an entangled pair between two qubits in a quantum computer by creating the Bell state $|\Phi^+\rangle = \frac{|00\rangle + |11\rangle}{\sqrt{2}}$. Using the Estimator primitive, you can directly obtain the expectation values needed ($\langle AB \rangle, \langle Ab \rangle, \langle aB \rangle$, and $\langle ab \rangle$) to calculate the expectation values of the two CHSH quantities $\langle S_1\rangle$ and $\langle S_2\rangle$. Before the introduction of the Estimator primitive, you would have to construct the expectation values from the measurement outcomes.

You will measure the second qubit in the $Z$ and $X$ bases.  The first qubit will be measured also in orthogonal bases, but with an angle with respect to the second qubit, which we are going to sweep between $0$ and $2\pi$. As you will see, the Estimator primitive makes running parameterized circuits very easy. Rather than creating a series of CHSH circuits, you only need to create *one* CHSH circuit with a parameter specifying the measurement angle and a series of phase values for the parameter.

Finally, you will analyze the results and plot them against the measurement angle. You will see that for certain range of measurement angles, the expectation values of CHSH quantities $|\langle S_1\rangle| > 2$ or $|\langle S_2\rangle| > 2$, which demonstrates the violation of the CHSH inequality.


In [1]:
from qiskit_aer import AerSimulator
from qiskit_ibm_runtime import Options, Session, SamplerV2 as Sampler
backend_sim = AerSimulator()
simulator = AerSimulator()

#Import an estimator, this time from qiskit (we import from Runtime for real hardware)
from qiskit.primitives import BackendSampler
sampler = BackendSampler(backend = backend_sim)
import numpy as np
from qiskit.visualization import plot_bloch_vector, plot_histogram
import matplotlib.pyplot as plt

from qiskit.circuit import QuantumRegister, ClassicalRegister, QuantumCircuit, Parameter
from qiskit import QuantumCircuit, transpile
from qiskit.result import marginal_counts
from qiskit.quantum_info import SparsePauliOp
from qiskit_ibm_runtime import EstimatorV2 as Estimator


import matplotlib.ticker as tck



/var/folders/8x/4b43jvsj34l67v7lt969s9bm0000gn/T/ipykernel_64028/3344170123.py:8: DeprecationWarning: The class ``qiskit.primitives.backend_sampler.BackendSampler`` is deprecated as of qiskit 1.2. It will be removed no earlier than 3 months after the release date. All implementations of the `BaseSamplerV1` interface have been deprecated in favor of their V2 counterparts. The V2 alternative for the `BackendSampler` class is `BackendSamplerV2`.
  sampler = BackendSampler(backend = backend_sim)


In [26]:
backend = AerSimulator()

### Step 1 — The 4 measurements

Recall the CHSH correlators $\langle AB \rangle$, $\langle Ab \rangle$, $\langle aB \rangle$, $\langle ab \rangle$.  The four measurement settings are:

**Alice (qubit 0)** — two orthogonal bases:
- $A = X$
- $a = Z$

**Bob (qubit 1)** — two bases rotated by $\theta$ w.r.t. Alice:
- $B = \cos\theta \, Z + \sin\theta \, X$
- $b = -\sin\theta \, Z + \cos\theta \, X$  (orthogonal to $B$)

Note that $A$ and $a$ are simply $X$ and $Z$, while $B$ and $b$ are continuous rotations controlled by $\theta$.  By sweeping $\theta \in [0, 2\pi]$ we explore the full range of correlations.

**Exercise 1:** For which values of $\theta$ do you expect the CHSH inequality to be violated (i.e. $|S_1| > 2$ or $|S_2| > 2$)?

*Hint:* The quantum maximum is $2\sqrt{2}$ (Tsirelson's bound), reached at specific angles. Try to work out analytically what $S_1(\theta)$ looks like for the Bell state $|\Phi^+\rangle$.

###  How to measure $B = \cos\theta\,Z + \sin\theta\,X$

A quantum computer can only measure in the $Z$ basis natively.  To measure a rotated observable we **rotate the qubit first**, then measure $Z$.

The key identity is:
$$R_y(-\alpha)\,|\psi\rangle \xrightarrow{\text{measure }Z} \text{same statistics as measuring }(\cos\alpha\,Z + \sin\alpha\,X)\text{ on }|\psi\rangle$$

So to measure $B = \cos\theta\,Z + \sin\theta\,X$ on Bob, we apply $R_y(-\theta)$ before the $Z$ measurement.  
Similarly, $b = -\sin\theta\,Z + \cos\theta\,X$ requires $R_y(-\theta - \pi/2)$.

For Alice, $A = X$ requires $R_y(-\pi/2)$, while $a = Z$ requires no rotation at all.

The circuit below shows Bob's qubit prepared in some state, followed by $R_y(-\theta)$ to measure in the $B$ basis:

### Step 2 — The two CHSH witnesses

The two CHSH quantities are:
$$S_1 = \langle AB \rangle - \langle Ab \rangle + \langle aB \rangle + \langle ab \rangle$$
$$S_2 = \langle AB \rangle + \langle Ab \rangle - \langle aB \rangle + \langle ab \rangle$$

**Exercise 1:** At which angle $\theta$ do you expect $|S_1|$ or $|S_2|$ to be maximised? What is the theoretical maximum (Tsirelson bound)?  
*Hint: compute $S_1(\theta)$ analytically for the Bell state $|\Phi^+\rangle$.*

### Step 3 — Simulate at the optimal angle

Report $\langle AB\rangle$, $\langle Ab\rangle$, $\langle aB\rangle$, $\langle ab\rangle$, $S_1$, $S_2$, and whether $|S_1| > 2$ or $|S_2| > 2$.

**Exercise 3:** Using the 4 circuits you built above, evaluate the CHSH correlators at the angle that maximises Bell violation (from Exercise 1). Use the noiseless `AerSimulator`.

A correlator from shot counts is $E = (N_{\text{even}} - N_{\text{odd}}) / N_{\text{total}}$, where even parity ($00$, $11$) gives $+1$ and odd parity ($01$, $10$) gives $-1$.

# Exercise
#### 1 - Build the circuits for the 4 measurements


In [3]:
# AB
qc_AB = QuantumCircuit(2,2)
# A = X measurement on qubit 0
qc_AB.h(0)

# B = cos(theta) * Z + sin(theta) * X measurement on qubit 1
theta = Parameter('θ')
qc_AB.ry(-theta, 1)
qc_AB.measure_all()
# display(qc_AB.draw('mpl',))

# Ab
qc_Ab = QuantumCircuit(2,2)
# A = X measurement on qubit 0
qc_Ab.h(0)
# b = cos(theta) * Z - sin(theta) * X measurement on qubit 1
qc_Ab.ry(-theta + np.pi/2 , 1)

qc_Ab.measure_all()
# display(qc_Ab.draw('mpl',))


#aB
# Z measurement on qubit 0
qc_aB = QuantumCircuit(2,2)
qc_aB.ry(-theta, 1)
qc_aB.measure_all()
# display(qc_aB.draw('mpl',)) 


#ab
qc_ab = QuantumCircuit(2,2)
qc_ab.ry(-theta + np.pi/2, 1)
qc_ab.measure_all()
# display(qc_ab.draw('mpl',))

# Define the Bell state
qc_bell = QuantumCircuit(2)
qc_bell.h(0)
qc_bell.cx(0, 1)
# qc_bell.draw('mpl',)




#### 2- Compute the 4 correlators for the maximal violation angle


In [7]:
# Your code here
# Optimal angle for maximal CHSH violation
theta_opt = np.pi / 4
shots = 1000

# Compose Bell state with each measurement circuit
qc_AB_full = qc_bell.compose(qc_AB)
qc_Ab_full = qc_bell.compose(qc_Ab)
qc_aB_full = qc_bell.compose(qc_aB)
qc_ab_full = qc_bell.compose(qc_ab)

circuits = [qc_AB_full, qc_Ab_full, qc_aB_full, qc_ab_full]

# Bind the optimal angle and transpile
bound_circuits = [qc.assign_parameters({theta: theta_opt}) for qc in circuits]
compiled = [transpile(qc, backend) for qc in bound_circuits]

# Execute the circuits
results = [backend.run(qc, shots=shots).result() for qc in compiled]
counts = [res.get_counts() for res in results]


def correlator(counts):
    """Compute <ZZ> correlator from counts. Even parity (00,11) -> +1, odd (01,10) -> -1."""
    n_even = 0
    n_odd = 0
    for bitstring, count in counts.items():
        # Take only the first register (qubits), ignore ancilla bits if present
        bits = bitstring.replace(" ", "")
        # Count number of 1s in the two qubit bits
        parity = (bits.count('1')) % 2
        if parity == 0:
            n_even += count
        else:
            n_odd += count
    total = n_even + n_odd
    return (n_even - n_odd) / total

E_AB = correlator(counts[0])
E_Ab = correlator(counts[1])
E_aB = correlator(counts[2])
E_ab = correlator(counts[3])

S1 = E_AB - E_Ab + E_aB + E_ab
S2 = E_AB + E_Ab - E_aB + E_ab

print(f"<AB> = {E_AB:.4f}")
print(f"<Ab> = {E_Ab:.4f}")
print(f"<aB> = {E_aB:.4f}")
print(f"<ab> = {E_ab:.4f}")
print(f"\nS1 = <AB> - <Ab> + <aB> + <ab> = {S1:.4f}")
print(f"S2 = <AB> + <Ab> - <aB> + <ab> = {S2:.4f}")
print(f"\nClassical bound: |S| <= 2")
print(f"Tsirelson bound: |S| <= 2√2 ≈ {2*np.sqrt(2):.4f}")
print(f"\n|S1| > 2: {abs(S1) > 2}  (|S1| = {abs(S1):.4f})")
print(f"|S2| > 2: {abs(S2) > 2}  (|S2| = {abs(S2):.4f})")

<AB> = 0.6940
<Ab> = -0.6540
<aB> = 0.6700
<ab> = 0.6860

S1 = <AB> - <Ab> + <aB> + <ab> = 2.7040
S2 = <AB> + <Ab> - <aB> + <ab> = 0.0560

Classical bound: |S| <= 2
Tsirelson bound: |S| <= 2√2 ≈ 2.8284

|S1| > 2: True  (|S1| = 2.7040)
|S2| > 2: False  (|S2| = 0.0560)


#### 4- Run it on a noisy machine (see 01_3) and check the violation is beyond statistical noise

In [10]:
# ============================================================================
# IN CASE NO QPU ACCESS - Use Noisy Simulator  
# ============================================================================
# Create a noisy simulator (digital twin) instead of using real hardware
from qiskit_ibm_runtime import QiskitRuntimeService
# Use a local fake backend (no Runtime service connection needed)
from qiskit_ibm_runtime.fake_provider import FakeFez

backend = FakeFez()
print(f"Using noise model from: {backend.name}")

# Create a noisy simulator using the backend's noise characteristics
from qiskit_aer import AerSimulator
backend = AerSimulator.from_backend(backend)

# Use BackendSampler with the noisy simulator
from qiskit.primitives import BackendSampler
sampler = Sampler(mode=backend)
sampler.options.default_shots = 100

from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm = generate_preset_pass_manager(optimization_level=3, backend=backend) 

Using noise model from: fake_fez


In [24]:
# ── Exercise 4: Run the 4 correlator circuits on the noisy backend ─────────
shots_noisy = 100
number_of_phases = 21
phases = np.linspace(0, 2 * np.pi, number_of_phases)

# Transpile the full circuits (Bell + measurement) for the noisy backend
qc_AB_t = pm.run(qc_AB_full)
qc_Ab_t = pm.run(qc_Ab_full)
qc_aB_t = pm.run(qc_aB_full)
qc_ab_t = pm.run(qc_ab_full)

param_sweep = phases

sampler_noisy = Sampler(mode=backend)
sampler_noisy.options.default_shots = shots_noisy

job = sampler_noisy.run([
    (qc_AB_t, param_sweep),
    (qc_Ab_t, param_sweep),
    (qc_aB_t, param_sweep),
    (qc_ab_t, param_sweep),
])

result = job.result()
print("Job complete!")

def correlator_and_sigma(pub_result, phase_idx):
    """Return (E, sigma_E) from a PubResult at a given phase index."""
    counts_dict = pub_result.data.meas[phase_idx].get_counts()
    n_even, n_odd = 0, 0
    for bitstring, count in counts_dict.items():
        bits = bitstring.replace(" ", "")
        parity = bits.count('1') % 2
        if parity == 0:
            n_even += count
        else:
            n_odd += count
    total = n_even + n_odd
    E = (n_even - n_odd) / total
    sigma = np.sqrt(1 - E**2) / np.sqrt(total)
    return E, sigma

chsh1_est = []
chsh2_est = []
sigma_S1  = []
sigma_S2  = []

res_AB = result[0]
res_Ab = result[1]
res_aB = result[2]
res_ab = result[3]

for i in range(number_of_phases):
    eAB, sAB = correlator_and_sigma(res_AB, i)
    eAb, sAb = correlator_and_sigma(res_Ab, i)
    eaB, saB = correlator_and_sigma(res_aB, i)
    eab, sab = correlator_and_sigma(res_ab, i)

    S1_val = eAB - eAb + eaB + eab
    S2_val = eAB + eAb - eaB + eab
    chsh1_est.append(S1_val)
    chsh2_est.append(S2_val)
    sigma_S1.append(np.sqrt(sAB**2 + sAb**2 + saB**2 + sab**2))
    sigma_S2.append(np.sqrt(sAB**2 + sAb**2 + saB**2 + sab**2))

chsh1_est = np.array(chsh1_est)
chsh2_est = np.array(chsh2_est)
sigma_S1  = np.array(sigma_S1)
sigma_S2  = np.array(sigma_S2)

print("S1 values:", np.round(chsh1_est, 3))
print("S2 values:", np.round(chsh2_est, 3))


Job complete!
S1 values: [ 1.92  2.62  2.62  2.66  2.24  1.86  1.08  0.4  -0.46 -1.26 -2.32 -2.4
 -2.68 -2.5  -2.44 -1.8  -1.22 -0.2   0.3   1.28  1.96]
S2 values: [-1.96 -1.14 -0.78  0.62  1.44  2.02  2.4   2.68  2.5   2.22  1.68  1.32
  0.48 -0.46 -1.4  -2.12 -2.46 -3.04 -2.86 -2.44 -1.96]


### Step 5 — Sweep $\theta$ and plot the CHSH witnesses

Plot $S_1$ and $S_2$ vs $\theta$ with error bars. Add horizontal lines for the classical bound $\pm 2$ and shade the region between the classical and Tsirelson ($\pm 2\sqrt{2}$) bounds. At which angles is the inequality violated?

**Exercise 5:** Sweep $\theta \in [0, 2\pi]$ (use `number_of_phases = 21` points) on the noisy backend. For each angle compute $S_1(\theta)$, $S_2(\theta)$ and their statistical uncertainties $\sigma_{S_1}$, $\sigma_{S_2}$.

In [ ]:
# ── Exercise 5 ────────────────────────────────────────────────────────────
# Sweep theta on the noisy backend and plot S1, S2 with error bars.

# param_sweep = [[float(ph)] for ph in phases]

# Hint: run all 4 circuits × all phases in one sampler.run() call:
# sampler_noisy = Sampler(mode=noisy_backend)
# sampler_noisy.options.default_shots = shots_noisy
# job = sampler_noisy.run([
#     (qc_AB_t, param_sweep),
#     (qc_Ab_t, param_sweep),
#     (qc_aB_t, param_sweep),

#     (qc_ab_t, param_sweep),

# ])plt.show()

plt.tight_layout()

# Extract counts per phase, compute correlators + sigmas, form S1 and S2.ax.legend()

# chsh1_est = ...ax.set_title("CHSH witnesses vs $\\theta$ — noisy backend")

# chsh2_est = ...ax.set_ylabel("CHSH witness")

# sigma_S1  = ...ax.set_xlabel(r"$\theta$")

# sigma_S2  = ...ax.xaxis.set_major_locator(tck.MultipleLocator(base=0.5))

ax.xaxis.set_major_formatter(tck.FormatStrFormatter("%g $\\pi$"))

fig, ax = plt.subplots(figsize=(10, 6))

ax.fill_between(phases / np.pi, -2, -2*np.sqrt(2), color="0.7", alpha=0.5)

# ax.errorbar(phases / np.pi, chsh1_est, yerr=sigma_S1, fmt="o-", capsize=4, label="S1")ax.fill_between(phases / np.pi,  2,  2*np.sqrt(2), color="0.7", alpha=0.5, label="Quantum violation")

# ax.errorbar(phases / np.pi, chsh2_est, yerr=sigma_S2, fmt="s-", capsize=4, label="S2")ax.axhline(y=-2*np.sqrt(2), color="0.5", linestyle="-.", linewidth=1)

ax.axhline(y= 2*np.sqrt(2), color="0.5", linestyle="-.", linewidth=1)

# Classical bound ±2# Tsirelson bound ±2√2

ax.axhline(y= 2, color="0.5", linestyle="--", linewidth=1, label="Classical bound ±2")
ax.axhline(y=-2, color="0.5", linestyle="--", linewidth=1)